In [1]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import _name_estimators
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
class MajorityVoteClassifier(BaseEstimator, ClassifierMixin):

    def __init__(self, classifiers, vote='classlabel', weights=None):
        self.classifiers = classifiers
        self.vote = vote
        self.weights = weights
        self.named_classifiers = {
            key: value
            for key, value in _name_estimators(classifiers)
        }

    def fit(self, X, y):

        if self.vote not in ('classlabel', 'probability'):
            raise ValueError(
                "vote must be 'classlabel' or 'probability'"
            )

        if self.weights is not None and \
                len(self.weights) != len(self.classifiers):
            raise ValueError(
                "Number of classifiers and weights must be equal."
            )

        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(y)

        self.classes_ = self.label_encoder.classes_

        self.classifiers_ = []

        for clf in self.classifiers:

            fitted_clf = clone(clf)

            fitted_clf.fit(
                X,
                self.label_encoder.transform(y)
            )

            self.classifiers_.append(fitted_clf)

        return self

    def predict(self, X):

        if self.vote == 'probability':

            maj_vote = np.argmax(
                self.predict_proba(X),
                axis=1
            )

        else:

            predictions = np.asarray(
                [
                    clf.predict(X)
                    for clf in self.classifiers_
                ]
            ).T

            maj_vote = np.apply_along_axis(

                lambda x:
                np.argmax(
                    np.bincount(
                        x,
                        weights=self.weights
                    )
                ),

                axis=1,

                arr=predictions

            )

        maj_vote = self.label_encoder.inverse_transform(
            maj_vote
        )

        return maj_vote

    def predict_proba(self, X):

        probas = np.asarray(
            [
                clf.predict_proba(X)
                for clf in self.classifiers_
            ]
        )

        avg_proba = np.average(
            probas,
            axis=0,
            weights=self.weights
        )

        return avg_proba

    def get_params(self, deep=True):

        if not deep:
            return super().get_params(deep=False)

        out = self.named_classifiers.copy()

        for name, clf in self.named_classifiers.items():

            for key, value in clf.get_params(
                    deep=True).items():

                out[f"{name}__{key}"] = value

        return out


X, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


clf1 = LogisticRegression(random_state=42)

clf2 = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

clf3 = KNeighborsClassifier(
    n_neighbors=5
)


mv_clf = MajorityVoteClassifier(

    classifiers=[
        clf1,
        clf2,
        clf3
    ],

    vote='classlabel'
)

mv_clf.fit(
    X_train,
    y_train
)


ensemble_predictions = mv_clf.predict(
    X_test
)

ensemble_accuracy = accuracy_score(
    y_test,
    ensemble_predictions
)

print("=" * 60)
print("Majority Vote Classifier")
print("=" * 60)

print("Accuracy :",
      round(ensemble_accuracy, 4))

print()

print(classification_report(
    y_test,
    ensemble_predictions
))


print("=" * 60)
print("Individual Classifiers")
print("=" * 60)

models = {

    "Logistic Regression": clf1,

    "Decision Tree": clf2,

    "KNN": clf3

}

for name, model in models.items():

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_test
    )

    acc = accuracy_score(
        y_test,
        prediction
    )

    print(f"{name:<22}: {acc:.4f}")

print()

print(f"{'Majority Voting':<22}: {ensemble_accuracy:.4f}")

Majority Vote Classifier
Accuracy : 0.885

              precision    recall  f1-score   support

           0       0.86      0.94      0.90       109
           1       0.93      0.81      0.87        91

    accuracy                           0.89       200
   macro avg       0.89      0.88      0.88       200
weighted avg       0.89      0.89      0.88       200

Individual Classifiers
Logistic Regression   : 0.7000
Decision Tree         : 0.8750
KNN                   : 0.8900

Majority Voting       : 0.8850
